# 02 - Logistic Regression Baseline
**Project:** Credit Card Default Prediction  
**Model:** Logistic Regression (interpretable baseline)

---
### Environment Setup
- **VS Code / Local:** Data path is `../data/UCI_Credit_Card.csv` (default, no changes needed)
- **Google Colab:** Upload the CSV manually, then change `DATA_PATH` to `'UCI_Credit_Card.csv'`

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# Change to 'UCI_Credit_Card.csv' if using Google Colab
DATA_PATH = '../data/UCI_Credit_Card.csv'

df = pd.read_csv(DATA_PATH)
df.rename(columns={'default payment next month': 'default'}, inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["default"].mean():.2%}')

In [ ]:
# ── Feature / Target Split ────────────────────────────────────────────────────
# Drop ID column (not a predictive feature) and separate target
X = df.drop(columns=['ID', 'default'])
y = df['default']

print(f'Features: {X.shape[1]}')
print(f'Samples:  {X.shape[0]}')

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────────
# Use stratified split to preserve class ratio in both sets
# 80% train, 20% test — fixed random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train set: {X_train.shape[0]} samples | Default rate: {y_train.mean():.2%}')
print(f'Test set:  {X_test.shape[0]} samples  | Default rate: {y_test.mean():.2%}')

In [ ]:
# ── Feature Scaling ───────────────────────────────────────────────────────────
# Logistic Regression is sensitive to feature scale — StandardScaler normalizes
# each feature to mean=0, std=1
# IMPORTANT: fit only on training data, then apply to test data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling applied: mean=0, std=1 for all features')

In [ ]:
# ── Train Logistic Regression ─────────────────────────────────────────────────
# class_weight='balanced' adjusts for class imbalance (~22% default)
# max_iter=1000 ensures convergence
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_scaled, y_train)

print('Model training complete.')

In [ ]:
# ── Cross-Validation (Best Practice) ─────────────────────────────────────────
# 5-fold stratified CV on training data to check model stability
# This guards against overfitting to a single train/test split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
cv_recall = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='recall')

print('=== 5-Fold Cross-Validation Results (Train Set) ===')
print(f'AUC-ROC : {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})')
print(f'Recall  : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})')

In [ ]:
# ── Evaluate on Test Set ──────────────────────────────────────────────────────
y_pred      = lr_model.predict(X_test_scaled)
y_pred_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print('=== Test Set Performance ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred):.4f}')
print(f'AUC-ROC   : {roc_auc_score(y_test, y_pred_prob):.4f}')
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Default', 'Default'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=ax, name='Logistic Regression')
ax.plot([0, 1], [0, 1], 'k--', label='Random baseline')
ax.set_title('ROC Curve — Logistic Regression')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance (Coefficients) ────────────────────────────────────────
# In logistic regression, the magnitude of coefficients indicates feature importance
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['salmon' if c > 0 else 'steelblue' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.set_title('Top 15 Feature Coefficients — Logistic Regression')
ax.set_xlabel('Coefficient Value (Red = increases default risk)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Save Model ────────────────────────────────────────────────────────────────
# Save trained model and scaler for reuse in later notebooks
joblib.dump(lr_model, '../models/logistic_regression.pkl')
joblib.dump(scaler,   '../models/scaler.pkl')

print('Model saved to: models/logistic_regression.pkl')
print('Scaler saved to: models/scaler.pkl')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=== Logistic Regression Baseline Summary ===')
print(f'AUC-ROC  : {roc_auc_score(y_test, y_pred_prob):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print('\nThis baseline will be compared against Random Forest and XGBoost in notebook 03.')